# Beyond Combinations
## Investigating Visual Generalization to Unseen Attribute–Object Combinations

### Project Overview

This project investigates whether deep learning models can recognize familiar visual attributes when they appear in combinations that were not observed during training.

The experiment uses the **CLEVR-CoGenT v1.0** dataset to study compositional generalization in visual recognition.

### Research Question

Can a model trained on familiar shape–color combinations correctly recognize unseen combinations of those same shapes and colors?

### Experimental Design

| Stage | Dataset | Purpose |
|---|---|---|
| Training | `trainA` | Train models on Condition A |
| Validation | `valA` | Evaluate performance on familiar combinations |
| Evaluation | Selected objects from `valB` | Test generalization to unseen combinations |

### Models

1. **Joint Classifier:** Predicts each observed shape–color combination as a single class.
2. **Factorized Classifier:** Uses a shared feature extractor with separate prediction heads for shape and color.

### Main Evaluation

- Shape accuracy
- Color accuracy
- Joint shape–color accuracy
- Precision, recall, and F1-score
- Confusion matrices
- Generalization gap

### Workflow

1. Dataset inspection and validation
2. Attribute and combination analysis
3. Object crop extraction
4. Dataset persistence and quality checks
5. Model development
6. Evaluation and comparison

# 1. Environment Setup

This section imports the required libraries, defines the dataset and output paths, and configures reproducibility.

All preprocessing outputs will be stored in a dedicated working directory.

In [3]:
import os
import json
import csv
import random
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
from tqdm.auto import tqdm

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Image configuration
CROP_SIZE = 112
PADDING_COLOR = (127, 127, 127)
JPEG_QUALITY = 95

# Dataset paths
DATASET_PATH = Path(
    "/kaggle/input/datasets/maram05ai/"
    "clevr-cogent-v1/CLEVR_CoGenT_v1.0"
)

# Output directory
OUTPUT_PATH = Path("/kaggle/working/clevr_cogent_processed")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Verify dataset availability
assert DATASET_PATH.exists(), (
    f"Dataset not found: {DATASET_PATH}\n"
    "Make sure CLEVR-CoGenT is attached to this notebook."
)

print("Environment initialized successfully.")
print(f"Dataset path: {DATASET_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Crop size: {CROP_SIZE} × {CROP_SIZE}")
print(f"Random seed: {SEED}")

Environment initialized successfully.
Dataset path: /kaggle/input/datasets/maram05ai/clevr-cogent-v1/CLEVR_CoGenT_v1.0
Output path: /kaggle/working/clevr_cogent_processed
Crop size: 112 × 112
Random seed: 42


# 2. Dataset Overview

CLEVR-CoGenT is a synthetic visual reasoning dataset designed to investigate compositional generalization.

The dataset contains scenes with objects characterized by attributes such as:

- Shape
- Color
- Material
- Size

The experiment uses three splits:

- `trainA`: Training data from Condition A.
- `valA`: Validation data from Condition A.
- `valB`: Data from Condition B, used to evaluate unseen shape–color combinations.

We first load the scene annotations and inspect the dataset structure before processing any images.

In [4]:
def load_scenes(split):
    annotation_path = (
        DATASET_PATH
        / "scenes"
        / f"CLEVR_{split}_scenes.json"
    )

    if not annotation_path.exists():
        raise FileNotFoundError(
            f"Annotation file not found: {annotation_path}"
        )

    with open(annotation_path, "r") as file:
        data = json.load(file)

    return data["scenes"]


# Load annotations
train_scenes = load_scenes("trainA")
valA_scenes = load_scenes("valA")
valB_scenes = load_scenes("valB")

SPLITS = {
    "trainA": train_scenes,
    "valA": valA_scenes,
    "valB": valB_scenes
}

# Display scene counts
for split_name, scenes in SPLITS.items():
    print(f"{split_name}: {len(scenes):,} scenes")

trainA: 70,000 scenes
valA: 15,000 scenes
valB: 15,000 scenes


In [5]:
# Inspect one scene
sample_scene = train_scenes[0]

print("Scene keys:")
print(sample_scene.keys())

print("\nImage filename:")
print(sample_scene["image_filename"])

print("\nNumber of objects:")
print(len(sample_scene["objects"]))

print("\nFirst object annotation:")
print(json.dumps(
    sample_scene["objects"][0],
    indent=4
))

Scene keys:
dict_keys(['image_filename', 'objects', 'split', 'image_index', 'relationships', 'directions'])

Image filename:
CLEVR_trainA_000000.png

Number of objects:
3

First object annotation:
{
    "pixel_coords": [
        131,
        173,
        9.2485990524292
    ],
    "size": "large",
    "color": "yellow",
    "material": "metal",
    "shape": "sphere",
    "3d_coords": [
        0.044448722153902054,
        -2.6995933055877686,
        0.699999988079071
    ],
    "rotation": 303.2391460924107
}


In [6]:
for split_name in SPLITS.keys():

    image_dir = DATASET_PATH / "images" / split_name

    print(f"\n{split_name}")
    print("-" * 40)
    print("Image directory:", image_dir)
    print("Directory exists:", image_dir.exists())

    if image_dir.exists():
        image_count = len(list(image_dir.glob("*.png")))
        print(f"PNG images: {image_count:,}")


trainA
----------------------------------------
Image directory: /kaggle/input/datasets/maram05ai/clevr-cogent-v1/CLEVR_CoGenT_v1.0/images/trainA
Directory exists: True
PNG images: 70,000

valA
----------------------------------------
Image directory: /kaggle/input/datasets/maram05ai/clevr-cogent-v1/CLEVR_CoGenT_v1.0/images/valA
Directory exists: True
PNG images: 15,000

valB
----------------------------------------
Image directory: /kaggle/input/datasets/maram05ai/clevr-cogent-v1/CLEVR_CoGenT_v1.0/images/valB
Directory exists: True
PNG images: 15,000


# 3. Data Integrity Validation

Before extracting object crops, we validate the relationship between scene annotations and image files.

The validation checks:

1. Whether every annotated scene has a corresponding image.
2. Whether object annotations contain pixel coordinates.
3. Whether a sample of source images can be opened successfully.

This step helps detect missing files and malformed annotations before preprocessing.

In [7]:
def get_image_path(split_name, scene):
    return (
        DATASET_PATH
        / "images"
        / split_name
        / scene["image_filename"]
    )


# Test the helper
sample_path = get_image_path("trainA", train_scenes[0])

print("Sample image path:")
print(sample_path)

print("Exists:", sample_path.exists())

Sample image path:
/kaggle/input/datasets/maram05ai/clevr-cogent-v1/CLEVR_CoGenT_v1.0/images/trainA/CLEVR_trainA_000000.png
Exists: True


In [8]:
def validate_split(split_name, scenes, sample_size=100):

    missing_images = []
    malformed_objects = []
    unreadable_images = []

    # Check all image paths and object coordinates
    for scene_idx, scene in enumerate(
        tqdm(scenes, desc=f"Checking {split_name}")
    ):

        image_path = get_image_path(split_name, scene)

        if not image_path.exists():
            missing_images.append(scene["image_filename"])

        for object_idx, obj in enumerate(scene["objects"]):

            coords = obj.get("pixel_coords")

            if (
                coords is None
                or len(coords) < 2
                or not all(
                    isinstance(value, (int, float))
                    for value in coords[:2]
                )
            ):
                malformed_objects.append(
                    (scene_idx, object_idx)
                )

    # Verify a sample of images can be opened
    sample = scenes[:min(sample_size, len(scenes))]

    for scene in tqdm(
        sample,
        desc=f"Reading image sample: {split_name}"
    ):

        image_path = get_image_path(split_name, scene)

        if not image_path.exists():
            continue

        try:
            with Image.open(image_path) as image:
                image.verify()

        except Exception as error:
            unreadable_images.append(
                (scene["image_filename"], str(error))
            )

    return {
        "split": split_name,
        "total_scenes": len(scenes),
        "missing_images": len(missing_images),
        "malformed_objects": len(malformed_objects),
        "unreadable_sample_images": len(unreadable_images),
        "missing_examples": missing_images[:5],
        "malformed_examples": malformed_objects[:5],
        "unreadable_examples": unreadable_images[:5]
    }


integrity_reports = []

for split_name, scenes in SPLITS.items():

    report = validate_split(split_name, scenes)

    integrity_reports.append(report)


integrity_summary = pd.DataFrame([
    {
        "Split": report["split"],
        "Scenes": report["total_scenes"],
        "Missing Images": report["missing_images"],
        "Malformed Objects": report["malformed_objects"],
        "Unreadable Sample Images":
            report["unreadable_sample_images"]
    }
    for report in integrity_reports
])

display(integrity_summary)

Checking trainA:   0%|          | 0/70000 [00:00<?, ?it/s]

Reading image sample: trainA:   0%|          | 0/100 [00:00<?, ?it/s]

Checking valA:   0%|          | 0/15000 [00:00<?, ?it/s]

Reading image sample: valA:   0%|          | 0/100 [00:00<?, ?it/s]

Checking valB:   0%|          | 0/15000 [00:00<?, ?it/s]

Reading image sample: valB:   0%|          | 0/100 [00:00<?, ?it/s]

,Split,Scenes,Missing Images,Malformed Objects,Unreadable Sample Images
0,trainA,70000,0,0,0
1,valA,15000,0,0,0
2,valB,15000,0,0,0


In [9]:
for report in integrity_reports:

    print(f"\n{report['split']}")
    print("=" * 50)

    print("Missing image examples:")
    print(report["missing_examples"])

    print("\nMalformed object examples:")
    print(report["malformed_examples"])

    print("\nUnreadable image examples:")
    print(report["unreadable_examples"])


trainA
Missing image examples:
[]

Malformed object examples:
[]

Unreadable image examples:
[]

valA
Missing image examples:
[]

Malformed object examples:
[]

Unreadable image examples:
[]

valB
Missing image examples:
[]

Malformed object examples:
[]

Unreadable image examples:
[]


# 4. Attribute and Combination Analysis

This section examines the distribution of object attributes across the three dataset splits.

Each object is described by several attributes, including shape, color, material, and size.

Our main focus is the relationship between shape and color because the experiment investigates generalization to unseen shape–color combinations.

We will:

1. Extract object-level attribute information from scene annotations.
2. Calculate shape and color distributions.
3. Count the observed shape–color combinations.
4. Identify the eight combinations that are unseen in Condition A and evaluated in Condition B.

In [10]:
def extract_object_records(scenes):

    records = []

    for scene_idx, scene in enumerate(scenes):

        for object_idx, obj in enumerate(scene["objects"]):

            records.append({
                "scene_index": scene_idx,
                "object_index": object_idx,
                "shape": obj.get("shape"),
                "color": obj.get("color"),
                "material": obj.get("material"),
                "size": obj.get("size")
            })

    return pd.DataFrame(records)


train_objects = extract_object_records(train_scenes)
valA_objects = extract_object_records(valA_scenes)
valB_objects = extract_object_records(valB_scenes)


print("TrainA objects:", f"{len(train_objects):,}")
print("ValA objects:", f"{len(valA_objects):,}")
print("ValB objects:", f"{len(valB_objects):,}")

display(train_objects.head())

TrainA objects: 455,024
ValA objects: 97,900
ValB objects: 97,193


,scene_index,object_index,shape,color,material,size
0,0,0,sphere,yellow,metal,large
1,0,1,cylinder,cyan,metal,large
2,0,2,sphere,gray,rubber,small
3,1,0,cube,blue,metal,large
4,1,1,sphere,blue,rubber,large


In [11]:
for split_name, dataframe in {
    "trainA": train_objects,
    "valA": valA_objects,
    "valB": valB_objects
}.items():

    print(f"\n{split_name} Attribute Distribution")
    print("=" * 50)

    print("\nShape counts:")
    display(
        dataframe["shape"]
        .value_counts()
        .rename_axis("Shape")
        .reset_index(name="Count")
    )

    print("\nColor counts:")
    display(
        dataframe["color"]
        .value_counts()
        .rename_axis("Color")
        .reset_index(name="Count")
    )


trainA Attribute Distribution

Shape counts:


,Shape,Count
0,sphere,152744
1,cylinder,151544
2,cube,150736



Color counts:


,Color,Count
0,green,57226
1,red,57055
2,brown,57001
3,cyan,56951
4,yellow,56830
5,purple,56770
6,gray,56735
7,blue,56456



valA Attribute Distribution

Shape counts:


,Shape,Count
0,sphere,33315
1,cylinder,32714
2,cube,31871


... (2 outputs left out)


valB Attribute Distribution

Shape counts:


,Shape,Count
0,cube,32450
1,cylinder,32380
2,sphere,32363



Color counts:


,Color,Count
0,green,12317
1,red,12245
2,gray,12159
3,blue,12110
4,brown,12104
5,purple,12086
6,cyan,12086
7,yellow,12086


In [12]:
def get_combination_counts(dataframe):

    return (
        dataframe
        .groupby(["shape", "color"])
        .size()
        .reset_index(name="count")
        .sort_values(["shape", "color"])
    )


for split_name, dataframe in {
    "trainA": train_objects,
    "valA": valA_objects,
    "valB": valB_objects
}.items():

    print(f"\n{split_name} Shape–Color Combinations")
    print("=" * 50)

    display(get_combination_counts(dataframe))


trainA Shape–Color Combinations


,shape,color,count
0,cube,blue,37341
1,cube,brown,37819
2,cube,gray,37634
3,cube,yellow,37942
4,cylinder,cyan,37820
5,cylinder,green,38130
6,cylinder,purple,37633
7,cylinder,red,37961
8,sphere,blue,19115
9,sphere,brown,19182



valA Shape–Color Combinations


,shape,color,count
0,cube,blue,8008
1,cube,brown,7875
2,cube,gray,8033
3,cube,yellow,7955
4,cylinder,cyan,8092
5,cylinder,green,8253
6,cylinder,purple,8270
7,cylinder,red,8099
8,sphere,blue,4132
9,sphere,brown,4143



valB Shape–Color Combinations


,shape,color,count
0,cube,cyan,8044
1,cube,green,8225
2,cube,purple,7997
3,cube,red,8184
4,cylinder,blue,8074
5,cylinder,brown,8099
6,cylinder,gray,8071
7,cylinder,yellow,8136
8,sphere,blue,4036
9,sphere,brown,4005


In [13]:
# Shape-color combinations absent from Condition A
# and evaluated in Condition B.

UNSEEN_PAIRS = {
    ("cube", "cyan"),
    ("cube", "green"),
    ("cube", "purple"),
    ("cube", "red"),

    ("cylinder", "blue"),
    ("cylinder", "brown"),
    ("cylinder", "gray"),
    ("cylinder", "yellow")
}

print(f"Number of unseen combinations: {len(UNSEEN_PAIRS)}")

for shape, color in sorted(UNSEEN_PAIRS):
    print(f"{shape} + {color}")

Number of unseen combinations: 8
cube + cyan
cube + green
cube + purple
cube + red
cylinder + blue
cylinder + brown
cylinder + gray
cylinder + yellow
